In [ ]:
PINECONE_API_KEY=""
GOOGLE_API_KEY=""

In [2]:
!pip install "langchain-core<1.0.0" "langchain<1.0.0"
#!pip install langchain==0.3.26
!pip install langchain-core==0.3.86
!pip install sentence-transformers==4.1.0
!pip install pypdf==5.6.1
#!pip install python-dotenv==1.1.0
!pip install langchain-pinecone==0.2.8
!pip install langchain-community==0.3.26
!pip install langchain-google-genai==2.1.5

  Using cached sentence_transformers-4.1.0-py3-none-any.whl.metadata (13 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached sentence_transformers-4.1.0-py3-none-any.whl (345 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.27.0
    Uninstalling huggingface_hub-1.27.0:
      Successfully uninstalled huggingface_hub-1.27.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.0
    Uninstalling transformers-5.15.0:
      Successfully uninstalled transformers-5.15.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0
ERROR: pip'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.9 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.6 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


#Loading the Text document

In [3]:
from langchain.document_loaders import TextLoader, DirectoryLoader

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
def load_txt_files(data):
  loader = DirectoryLoader(data,glob="*.txt",loader_cls=TextLoader)
  documents = loader.load()
  return documents

In [6]:
extracted_docs = load_txt_files( '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge')

In [7]:
extracted_docs

[Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge/dataset_overview.txt'}, page_content='DATASET OVERVIEW\n================================================================\nSource: Olist Brazilian E-Commerce dataset (olist_master_dataset.csv),\na merged order/customer/product/payment/review table used to power the\nAI-Powered E-Commerce Analytics Platform (dashboards, forecasting,\nsentiment analysis, recommendations, and the GenAI insights/chat layer).\n\nRAW SHAPE\n- Rows: 119,143 (order-item level rows; one row per item within an order)\n- Columns: 37\n- Unique orders: 99,441\n- Unique customers (customer_unique_id): 96,096\n- Unique sellers: 3,095\n- Unique products: 32,951\n- Product categories (Portuguese, product_category_name): 73\n- Product categories (English, product_category_name_english): 71\n- Date range: order_purchase_timestamp spans 2016-09-04 to 2018-10-17\n  (bulk of volume is Jan 2017 - Aug 2018; Sept/Oct 2016 an

In [8]:
# to remove unwanted things from document, we want only source and contents
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs):
  minimal_docs = []
  for doc in docs:
    src = doc.metadata.get("source")
    minimal_docs.append(Document(page_content=doc.page_content,metadata={"source":src}))
  return minimal_docs

In [9]:
minimal_docs = filter_to_minimal_docs(extracted_docs)

In [10]:
minimal_docs

[Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge/dataset_overview.txt'}, page_content='DATASET OVERVIEW\n================================================================\nSource: Olist Brazilian E-Commerce dataset (olist_master_dataset.csv),\na merged order/customer/product/payment/review table used to power the\nAI-Powered E-Commerce Analytics Platform (dashboards, forecasting,\nsentiment analysis, recommendations, and the GenAI insights/chat layer).\n\nRAW SHAPE\n- Rows: 119,143 (order-item level rows; one row per item within an order)\n- Columns: 37\n- Unique orders: 99,441\n- Unique customers (customer_unique_id): 96,096\n- Unique sellers: 3,095\n- Unique products: 32,951\n- Product categories (Portuguese, product_category_name): 73\n- Product categories (English, product_category_name_english): 71\n- Date range: order_purchase_timestamp spans 2016-09-04 to 2018-10-17\n  (bulk of volume is Jan 2017 - Aug 2018; Sept/Oct 2016 an

#Chunking

In [11]:
#chunck size is character size,chunk over lap is if any word is cut ,then 20 characters take back so that is over lapping
from langchain import text_splitter

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [12]:
texts_chunk = text_split(minimal_docs)

print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 101


In [14]:
texts_chunk[91]

Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge/recommendation_results.txt'}, page_content="OPERATIONAL NOTES\n- All five artifacts (product_catalog.pkl, knn_content_model.pkl,\n  combined_features.pkl, knn_svd_model.pkl, latent_matrix.pkl) are\n  loaded together via joblib and cached with @st.cache_resource so the\n  models are only loaded once per session.\n- get_recommendations() returns an empty DataFrame if the requested\n  product_id isn't found in the catalog (defensive check before indexing).\n- No formal offline recommendation-quality metrics (precision@k,")

In [15]:
texts_chunk[82]

Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge/customer_analysis.txt'}, page_content="SELLER-SIDE INSIGHT LINKED TO CUSTOMER BEHAVIOR\nOrders dispatched with a high margin of safety (seller_shipping_margin\n> 2 days, i.e. sellers ship well ahead of their shipping_limit_date)\nachieve a 90-day repeat purchase rate 2.4x higher than orders where the\nseller's dispatch was delayed — seller fulfillment speed is strongly\nlinked to whether a customer comes back.")

In [16]:
texts_chunk[66]

Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge/business_insights.txt'}, page_content='-> recommendation_results.txt (content-based KNN or SVD-based KNN)\n- "Where is revenue/growth concentrated and who are our best customers?"\n  -> sales_kpis.txt + customer_analysis.txt (RFM segmentation)\n- "Why is a specific order/customer/product underperforming?"\n  -> delivery_analysis.txt + product_analysis.txt (delivery delay,\n     freight cost, category-level patterns)')

#Embedding

In [17]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)

    return embeddings

embedding = download_embeddings()

/tmp/ipykernel_2446/3990462341.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [19]:
vector = embedding.embed_query("My name is Saumya")
vector

[-0.03242948651313782,
 -0.026726968586444855,
 -0.09655705094337463,
 0.055084798485040665,
 -0.0346921943128109,
 -0.0014045668067410588,
 0.09923527389764786,
 -0.029300181195139885,
 0.03288181498646736,
 -0.011342812329530716,
 -0.019423572346568108,
 -0.12379033118486404,
 0.11699382215738297,
 -0.06931715458631516,
 -0.003390415571630001,
 -0.032648973166942596,
 0.022889500483870506,
 0.021438319236040115,
 -0.05002500116825104,
 -0.1040554940700531,
 -0.013185836374759674,
 0.037890128791332245,
 -0.03929753601551056,
 -0.0005430199671536684,
 0.000646720640361309,
 0.0051664309576153755,
 -0.00939628854393959,
 0.010408121161162853,
 -0.009588573127985,
 -0.0023869050201028585,
 0.0349079892039299,
 0.04155813530087471,
 0.06975552439689636,
 0.030670715495944023,
 0.009193056263029575,
 0.0019219130044803023,
 -0.11699128895998001,
 0.014444316737353802,
 0.032484471797943115,
 0.03203878551721573,
 -0.024531463161110878,
 -0.05270322039723396,
 0.04118070378899574,
 -0.0217

In [20]:
len(vector)

384

In [21]:
import os
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [22]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [23]:
pc

In [24]:
from pinecone import ServerlessSpec

index_name = "chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384, #dimension of embeddings
        metric='cosine',
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [25]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

#Retriever

In [26]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [27]:
docsearch

In [29]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [30]:
retrieved_docs = retriever.invoke("what is the total revenue?")
retrieved_docs

[Document(id='1ad2ef66-3c17-4e83-8c3e-24aee7fe9d96', metadata={'source': '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge/customer_analysis.txt'}, page_content='Segment summary (customers, avg monetary, total revenue):\n- Loyal: 39,125 customers, avg R$234.96, total revenue R$9,192,893.60\n- Champions: 15,988 customers, avg R$421.31, total revenue R$6,735,977.02\n- At Risk: 32,917 customers, avg R$127.25, total revenue R$4,188,725.64\n- Lost: 8,066 customers, avg R$57.29, total revenue R$462,067.75'),
 Document(id='f17202c4-2bd5-447b-a39c-e2a82484b7e9', metadata={'source': '/content/drive/MyDrive/Colab Notebooks/IIT Palakkad Advanced AI/knowledge/sales_kpis.txt'}, page_content='REVENUE BY PRODUCT CATEGORY (top 10 by payment_value)\n1. bed_bath_table: R$1,743,999 (11,988 orders)\n2. health_beauty: R$1,662,964 (10,032 orders)\n3. computers_accessories: R$1,599,481 (8,150 orders)\n4. furniture_decor: R$1,443,964 (8,832 orders)\n5. watches_gifts: R$1,430,553 (6,21

#LLM generation

In [31]:
#from langchain_openai import ChatOpenAI
#chatModel = ChatOpenAI(model = "gpt-4o")
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
chatModel = ChatGoogleGenerativeAI(model = "gemini-3.5-flash")

In [32]:
chatModel.invoke('Hello')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': []}, id='run--01a01f45-9eb0-7410-b41d-87fd9730aea7-0', usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 189, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 178}})

In [34]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [35]:
system_prompt = (
    """
    You are a Business Assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer the
    question. If you don't know the answer say that you don't know.
    Use five sentences maximum and keep the answer concise.
    \n\n
    {context}
    """

)

In [36]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human","{input}"),
    ]
)

In [37]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [38]:
response = rag_chain.invoke({"input":"What is the most valued product?"})

In [39]:
print(response["answer"])

Based on the provided data, the most valued product category by revenue is **bed_bath_table**, which generated R$1,743,999 across 11,988 orders. The second most valued category is **health_beauty** with R$1,662,964, followed by **computers_accessories** with R$1,599,481. These top categories consistently lead the leaderboard in both revenue and overall order volume.


In [40]:
response = rag_chain.invoke({"input":"What are the recommendation to improve revenue?"})

In [41]:
print(response["answer"])

To improve revenue and mitigate concentration risks, the business should focus on geographic and product diversification. First, expanding marketing and sales efforts outside of SP, RJ, and MG will reduce vulnerability, as SP alone currently accounts for approximately 38% of total revenue. Second, promoting products beyond the top 10 categories (like *bed_bath_table* and *health_beauty*) will help decrease the 60%+ revenue dependency on a narrow product base. Finally, implementing targeted win-back campaigns for the large "At Risk" segment (32,917 customers) and retention programs for high-value "Champions" and "Loyal" customers can secure and grow existing revenue streams.
